In [ ]:
# SHAM_SECRETS_FALLBACK_V2 (2026-10-15)
# كاكل يرفض قراءة الأسرار (HTTP 400) في النسخ المرفوعة آلياً من المراقب.
# لا تغيّر شيئاً حين تعمل الأسرار. وإلا: GITHUB_TOKEN -> استنساخ عام بلا توكن،
# KAGGLE_* -> kaggle.json من مجموعة البيانات الخاصة sham-runtime-creds (تُضبط
# في متغيرات البيئة فوراً كي تجدها كل الوحدات)، وتيليجرام يبقى اختيارياً.
def _sham_secrets_fallback_v2():
    import glob, json, os, subprocess
    os.environ.setdefault("GIT_TERMINAL_PROMPT", "0")
    subprocess.run(["git", "config", "--global", "url.https://github.com/.insteadOf", "https://@github.com/"], check=False)

    def creds():
        for p in sorted(glob.glob("/kaggle/input/**/kaggle.json", recursive=True)):
            try:
                with open(p) as f:
                    d = json.load(f)
                if d.get("username") and d.get("key"):
                    return d
            except Exception:
                pass
        return {}

    c = creds()
    print("sham-runtime-creds:", "موجودة" if c else "غير موجودة في المدخلات")
    if c and not (os.environ.get("KAGGLE_USERNAME") and (os.environ.get("KAGGLE_KEY") or os.environ.get("KAGGLE_API_TOKEN"))):
        os.environ["KAGGLE_USERNAME"] = c["username"]
        os.environ["KAGGLE_KEY"] = c["key"]
        if c["key"].startswith("KGAT_"):
            os.environ["KAGGLE_API_TOKEN"] = c["key"]
    try:
        from kaggle_secrets import UserSecretsClient
    except Exception:
        return
    if getattr(UserSecretsClient.get_secret, "_sham_safe2", False):
        return
    orig = UserSecretsClient.get_secret

    def get_secret(self, label, *a, **k):
        try:
            return orig(self, label, *a, **k)
        except Exception:
            if label == "GITHUB_TOKEN":
                print("تنبيه: سرّ GITHUB_TOKEN غير متاح لهذا التشغيل — استنساخ عام بلا توكن.")
                return ""
            if label in ("KAGGLE_USERNAME", "KAGGLE_KEY"):
                d = creds()
                if d:
                    return d["username"] if label == "KAGGLE_USERNAME" else d["key"]
            raise

    get_secret._sham_safe2 = True
    UserSecretsClient.get_secret = get_secret

try:
    _sham_secrets_fallback_v2()
except Exception as _sf2_exc:
    print("SHAM_SECRETS_FALLBACK_V2 غير مفعّل:", type(_sf2_exc).__name__)


# شام — الدفتر الشامل (مقعد واحد بدل ثلاثة دفاتر)

دفتر واحد يحوي **كل خلايا** ثلاثة دفاتر: **المدرّب الحي** (زحف وجمع وترميز وتدريب)، و**المرحلة الأولى** (تدريب النص)، و**المرحلة الثانية** (الصورة والصوت مع النص). كل قسم تحته خلاياه كما هي في دفتره الأصلي.

كل جلسة تشغّل ما يصلح لوقتها: يدور الترتيب يومياً فينال كل قسم دوره، ويعمل قسم ثانٍ في الجلسة ذاتها فقط إن اتسعت ميزانيته الزمنية. خلايا القسم الذي ليس دوره تُتخطى بسطر واحد.

لا يكرّر ما تعمله دفاتر أخرى: دفتر الصورة يشغّل الصوت والفيديو بعده، والمسار A والمسار B والمحادثة تبقى كلٌّ في مقعدها (كاتب واحد لكل مجموعة بيانات).

- **المسرّع:** CPU فيُجدوَل يومياً، أو GPU فيُشغَّل يدوياً (كاكل يرفض جدولة دفتر عليه GPU).
- **Internet:** يجب أن يكون مفعّلاً.
- **الأسرار:** KAGGLE_USERNAME و KAGGLE_KEY و GITHUB_TOKEN. لا يحتاج أي Add Input.
- **الاستئناف:** كل قسم يستأنف تلقائياً من آخر نقطة منشورة.
- لتشغيل قسم بعينه يدوياً: غيّري السطر `FORCE_SECTIONS` في خلية الاختيار.


## 1) سحب الكود الحقيقي من GitHub

In [ ]:
import os
import sys
import subprocess
from kaggle_secrets import UserSecretsClient

GITHUB_TOKEN = UserSecretsClient().get_secret("GITHUB_TOKEN")
REPO_URL = f"https://{GITHUB_TOKEN}@github.com/jonsnow-org/Ttbik.git"
BRANCH = "claude/free-services-marketplace-h6rwk2"
CLONE_DIR = "/kaggle/working/Ttbik"

if not os.path.exists(CLONE_DIR):
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, CLONE_DIR], check=True)
    print("تم سحب الكود الحقيقي من مستودع GitHub بنجاح.")
else:
    subprocess.run(["git", "-C", CLONE_DIR, "pull"], check=True)
    print("الكود موجود بالفعل في هذه الجلسة — تم سحب أي تحديثات جديدة عليه.")

# أمان: git يحفظ رابط الاستنساخ (بما فيه GITHUB_TOKEN) حرفياً داخل
# .git/config -- وهذا المجلد يبقى ضمن نتاج (Output) هذه الجلسة، الذي قد
# يُستخدَم لاحقاً كمُدخَل (Notebook Output) لجلسة أخرى، أو يُشارَك بأي شكل.
# نزع التوكن من الرابط المحفوظ فور نجاح الاستنساخ يمنع تسربه عبر هذا
# المسار تماماً (ثغرة حقيقية اكتشفتها المالكة، 2026-09-21).
subprocess.run(["git", "-C", CLONE_DIR, "remote", "set-url", "origin", "https://github.com/jonsnow-org/Ttbik.git"], check=True)

CODE_DIR = os.path.join(CLONE_DIR, "ai-system", "colab", "sham_small")
assert os.path.exists(os.path.join(CODE_DIR, "model.py")), f"لم يتم العثور على model.py داخل {CODE_DIR}"
sys.path.insert(0, CODE_DIR)
print("كود ShamSmall الحقيقي جاهز في:", CODE_DIR)


## 2) تثبيت المكتبات

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "av", "soundfile", "tokenizers", "datasets", "kaggle"], check=False)
print("تم تجهيز المكتبات.")


## 3) ما يعمل اليوم

In [ ]:
import datetime
import os

import torch

import sham_inputs  # يثبّت إصلاحات التدريب المشتركة ويطبّق تحديثات الخلايا
from sham_chain import ALL_IN_ONE_STAGES, install_section_gate, pick_stages

CUDA = torch.cuda.is_available()
FORCE_SECTIONS = []   # مثال: ["sham_small_training"]  — الأقسام: sham_live_trainer, sham_small_training, sham_small_training_stage2_multimodal
STAGES_TODAY = FORCE_SECTIONS or pick_stages(datetime.date.today().toordinal(), 11.0, CUDA)
os.environ["SHAM_ACTIVE_SECTIONS"] = ",".join(STAGES_TODAY)
install_section_gate()
print("الجهاز:", "GPU" if CUDA else "CPU", "| الأقسام التي تعمل اليوم:", STAGES_TODAY)


---
# القسم: المدرّب الحي — زحف وجمع وترميز وتدريب
`sham_live_trainer` — خلاياه أدناه كما هي في دفتره الأصلي.

In [ ]:
# SHAM_SECTION: sham_live_trainer
get_ipython().run_line_magic("reset", "-f")   # قسم جديد = دفتر جديد: لا متغيرات من قسم سابق
import gc

gc.collect()
try:
    import torch
    torch.cuda.empty_cache()   # يحرّر ذاكرة GPU بين قسمين في جلسة واحدة
except Exception:
    pass


# شام — المدرّب الحي (زحف ← جمع ← ترميز ← تدريب ← تكرير ← حفظ)

تطوير لخلية المهندس: شبكة عنكبوتية من مصادر مجانية متنوعة (ويكيبيديا وأخواتها العربية، 9 تغذيات أخبار عربية، صور بأوصافها من ويكيميديا وناسا ومتحفين وOpenverse، نطق عربي من Lingua Libre ومجموعات الكلام المرخّصة، فيديو من ويكيميديا وناسا)، والجمع والتدريب يعملان في الوقت نفسه.

- **المسرّع:** GPU مستحسن (T4)، ويعمل على CPU أيضاً.
- **Internet:** يجب أن يكون مفعّلاً (Settings → Internet on).
- **الأسرار:** KAGGLE_USERNAME و KAGGLE_KEY و GITHUB_TOKEN (نفس بقية الدفاتر). لا يحتاج أي Add Input.
- **المخرجات:** النموذج إلى `sham-crawl-checkpoint`، ونصوص الزحف إلى `sham-crawl-corpus` — ومرحلة المحادثة تلتقطهما تلقائياً (إصلاح ← دمج محروس، وتذكّر).

In [ ]:
# SHAM_SECTION: sham_live_trainer
import os
import sys
import subprocess
from kaggle_secrets import UserSecretsClient

GITHUB_TOKEN = UserSecretsClient().get_secret("GITHUB_TOKEN")
REPO_URL = f"https://{GITHUB_TOKEN}@github.com/jonsnow-org/Ttbik.git"
BRANCH = "claude/free-services-marketplace-h6rwk2"
CLONE_DIR = "/kaggle/working/Ttbik"

if not os.path.exists(CLONE_DIR):
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, CLONE_DIR], check=True)
    print("تم سحب الكود الحقيقي من مستودع GitHub بنجاح.")
else:
    subprocess.run(["git", "-C", CLONE_DIR, "pull"], check=True)
    print("الكود موجود بالفعل في هذه الجلسة — تم سحب أي تحديثات جديدة عليه.")

# أمان: git يحفظ رابط الاستنساخ (بما فيه GITHUB_TOKEN) حرفياً داخل
# .git/config -- وهذا المجلد يبقى ضمن نتاج (Output) هذه الجلسة، الذي قد
# يُستخدَم لاحقاً كمُدخَل (Notebook Output) لجلسة أخرى، أو يُشارَك بأي شكل.
# نزع التوكن من الرابط المحفوظ فور نجاح الاستنساخ يمنع تسربه عبر هذا
# المسار تماماً (ثغرة حقيقية اكتشفتها المالكة، 2026-09-21).
subprocess.run(["git", "-C", CLONE_DIR, "remote", "set-url", "origin", "https://github.com/jonsnow-org/Ttbik.git"], check=True)

CODE_DIR = os.path.join(CLONE_DIR, "ai-system", "colab", "sham_small")
assert os.path.exists(os.path.join(CODE_DIR, "model.py")), f"لم يتم العثور على model.py داخل {CODE_DIR}"
sys.path.insert(0, CODE_DIR)
print("كود ShamSmall الحقيقي جاهز في:", CODE_DIR)


In [ ]:
# SHAM_SECTION: sham_live_trainer
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "av", "soundfile", "tokenizers"], check=False)
print("تم تجهيز المكتبات.")

## التشغيل
خلية واحدة تقوم بكل شيء حتى النشر. كل التطويرات اللاحقة تصل عبر المستودع دون إعادة استيراد.

In [ ]:
# SHAM_SECTION: sham_live_trainer
import sham_inputs  # يثبّت إصلاحات التدريب المشتركة (الجدولة، الربط التبايني)
import sham_live

result = sham_live.run()
print(result)

---
# القسم: المرحلة الأولى — تدريب النص الأساسي
`sham_small_training` — خلاياه أدناه كما هي في دفتره الأصلي.

In [ ]:
# SHAM_SECTION: sham_small_training
get_ipython().run_line_magic("reset", "-f")   # قسم جديد = دفتر جديد: لا متغيرات من قسم سابق
import gc

gc.collect()
try:
    import torch
    torch.cuda.empty_cache()   # يحرّر ذاكرة GPU بين قسمين في جلسة واحدة
except Exception:
    pass


# شام — أول دورة تدريب نصية حقيقية (ShamSmall)

هذا الدفتر يشغّل **تدريباً حقيقياً على بيانات عربية حقيقية**، وليس اختباراً تجريبياً. نطاقه محدد بوضوح: **النص فقط** في هذه الجولة الأولى (توليد وفهم نصي). تدريب الصورة والصوت خطوة لاحقة منفصلة تحتاج بيانات صور/صوت حقيقية لم نجمعها بعد — لا داعٍ لتعقيد هذه الجولة الأولى بها.

الكود نفسه مرفوع فعلاً على نفس مستودع GitHub الذي نعمل عليه (`jonsnow-org/Ttbik`، مجلد `ai-system/colab/sham_small`) — هذا الدفتر يسحبه مباشرة من هناك أول ما يعمل.

## قبل الضغط على "Run All" — خطوتان فقط مرة واحدة:

1. **أضف رمز وصول (Token) لحسابك على GitHub كـ Kaggle Secret** (لأن المستودع خاص وليس عاماً، فلا يمكن سحبه بلا تصريح):
   - من GitHub: `Settings → Developer settings → Personal access tokens → Generate new token` (يكفي صلاحية `repo` فقط للقراءة).
   - في Kaggle داخل هذا الدفتر: **Add-ons → Secrets → Add a new secret** — اجعل الاسم `GITHUB_TOKEN` والقيمة هي الرمز الذي أنشأته.
2. **فعّل الإنترنت واختر معالج رسومي (GPU):**
   - من القائمة الجانبية: **Settings → Internet → On**.
   - **Settings → Accelerator → GPU T4 x2** (أو أي GPU متاح).

بعد هاتين الخطوتين فقط، اضغط **Run All** ودع الدفتر يعمل من أوله لآخره.


> **الاستئناف تلقائي بالكامل (2026-09-24):** لا حاجة لأي «Add Input». في كل تشغيل يجلب الدفتر آخر نسخة من مجموعة بياناته هو عبر Kaggle API (`sham_inputs.py`) إن لم تكن مرفقة، ويكمل منها. إن كانت مرفقة يدوياً تُستخدم كما هي.

> **CPU أو GPU — اختيارك (2026-09-24):** الدفتر يعمل على الاثنين. إن ضبطتِه على GPU ونفد رصيد GPU المجاني، يُعيده منسّق الاستئناف تلقائياً على المعالج العادي (CPU) ويكمل من نفس النقطة، ثم يرجعه إلى GPU حين يتجدد الرصيد. وإن ضبطتِه على CPU فلا يُنقل إلى GPU أبداً.

### 1) سحب الكود الحقيقي من GitHub مباشرة

In [ ]:
# SHAM_SECTION: sham_small_training
import os
import sys
import subprocess
from kaggle_secrets import UserSecretsClient

GITHUB_TOKEN = UserSecretsClient().get_secret("GITHUB_TOKEN")
REPO_URL = f"https://{GITHUB_TOKEN}@github.com/jonsnow-org/Ttbik.git"
# هذا الكود لا يزال على فرع (branch) العمل الحالي، وليس على الفرع الرئيسي main بعد —
# إن دُمج لاحقاً إلى main يمكن حذف --branch هذا بأمان.
BRANCH = "claude/free-services-marketplace-h6rwk2"
CLONE_DIR = "/kaggle/working/Ttbik"

if not os.path.exists(CLONE_DIR):
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, CLONE_DIR], check=True)
    print("تم سحب الكود الحقيقي من مستودع GitHub بنجاح.")
else:
    # يجذب أي تحديث/إصلاح حقيقي جديد تم رفعه على GitHub بعد بدء هذه
    # الجلسة، دون الحاجة لإعادة تشغيل الجلسة أو إعادة استنساخ الكود من الصفر.
    subprocess.run(["git", "-C", CLONE_DIR, "pull"], check=True)
    print("الكود موجود بالفعل في هذه الجلسة — تم سحب أي تحديثات جديدة عليه.")

# أمان: git يحفظ رابط الاستنساخ (بما فيه GITHUB_TOKEN) حرفياً داخل
# .git/config -- وهذا المجلد يبقى ضمن نتاج (Output) هذه الجلسة، الذي قد
# يُستخدَم لاحقاً كمُدخَل (Notebook Output) لجلسة أخرى، أو يُشارَك بأي شكل.
# نزع التوكن من الرابط المحفوظ فور نجاح الاستنساخ يمنع تسربه عبر هذا
# المسار تماماً (ثغرة حقيقية اكتشفتها المالكة، 2026-09-21).
subprocess.run(["git", "-C", CLONE_DIR, "remote", "set-url", "origin", "https://github.com/jonsnow-org/Ttbik.git"], check=True)

CODE_DIR = os.path.join(CLONE_DIR, "ai-system", "colab", "sham_small")
assert os.path.exists(os.path.join(CODE_DIR, "model.py")), f"لم يتم العثور على model.py داخل {CODE_DIR}"
sys.path.insert(0, CODE_DIR)
print("كود ShamSmall الحقيقي جاهز في:", CODE_DIR)


### 2) تثبيت المكتبات الإضافية غير الموجودة افتراضياً على Kaggle

In [ ]:
# SHAM_SECTION: sham_small_training
# لا نفرض رقم نسخة محدداً عمداً: Kaggle يأتي غالباً بنسخة tokenizers
# حديثة ومتوافقة أصلاً مع مكتبة transformers المثبّتة مسبقاً هناك —
# فرض نسخة قديمة يدوياً يتسبب بتعارض غير ضروري معها (رغم أننا لا
# نستخدم transformers إطلاقاً هنا). هذه الخلية تتحقق أولاً فقط.
try:
    import tokenizers
    print(f"مكتبة tokenizers متوفرة مسبقاً (نسخة {tokenizers.__version__}) — لا حاجة للتثبيت.")
except ImportError:
    import subprocess
    subprocess.run(["pip", "install", "-q", "tokenizers"], check=True)
    print("تم تثبيت مكتبة tokenizers.")


### 2b) نقطة الاستئناف — تُجلب تلقائياً بلا أي مدخلات يدوية

In [ ]:
# SHAM_SECTION: sham_small_training
from sham_inputs import resume_text_lineage

# آخر نسخة من sham-checkpoint (مرفقة، أو تُنزَّل تلقائياً عبر Kaggle API) مع أداة تقسيم النص
# المحفوظة معها نفسها. يبحث أيضاً في الاسم القديم nova-small-checkpoint (قبل إعادة تسمية
# المجموعة إلى sham-checkpoint) فلا يُفقد أي تقدّم منشور تحت الاسم القديم. لا شيء تحت أي من
# الاسمين = أول تشغيل حقيقي، يبدأ من الصفر.
LINEAGE = resume_text_lineage(
    "sham-checkpoint", forks=["nova-small-checkpoint"], tokenizer_pattern="*tokenizer*.json",
)

### 3) جمع بيانات نصية عربية حقيقية

هذه الخلية تسحب (streaming، بدون تنزيل كامل يستهلك المساحة) شريحة حقيقية من **ويكيبيديا العربية** — أكبر مصدر نص عربي نظيف ومجاني متاح مباشرة. يمكنك زيادة `MAX_DOCUMENTS` لاحقاً بعد أن ترى الدفتر يعمل بنجاح من أوله لآخره على عيّنة أصغر.


In [ ]:
# SHAM_SECTION: sham_small_training
from data_acquisition import stream_hf_text_corpus, text_stream_position

MAX_DOCUMENTS = 20_000
WIKI = ("wikimedia/wikipedia", "20231101.ar")
# كل تشغيل يكمل من حيث توقّف السابق في ويكيبيديا بدل إعادة نفس أول المقالات إلى الأبد:
# الموضع محفوظ في text_stream_progress_stage1.json داخل مجموعة بيانات هذا المسار. التشغيلات القديمة قبل هذا
# الإصلاح لم تحفظ موضعاً وكانت تقرأ أول 20,000 مقالة دائماً، فنبدأ بعدها.
stream_skip = text_stream_position(*WIKI) or 20_000
corpus_dir = "/kaggle/working/corpus/wikipedia_ar"
_progress = "/kaggle/working/checkpoints/text_stream_progress_stage1.json"
corpus_files = stream_hf_text_corpus(
    dataset_name=WIKI[0], config_name=WIKI[1], text_field="text", output_dir=corpus_dir,
    max_documents=MAX_DOCUMENTS, skip=stream_skip, progress_path=_progress,
)
if not corpus_files:  # وصلنا لنهاية ويكيبيديا العربية كلها — دورة جديدة من البداية
    corpus_files = stream_hf_text_corpus(
        dataset_name=WIKI[0], config_name=WIKI[1], text_field="text", output_dir=corpus_dir,
        max_documents=MAX_DOCUMENTS, skip=0, progress_path=_progress,
    )
print(f"{len(corpus_files)} shard file(s) from article #{stream_skip:,} onward")

**اختياري:** إذا أردت إضافة معرفة شام الحالية الحقيقية (المحادثات وقاعدة المعرفة الموجودة فعلياً في Supabase) كبيانات تدريب إضافية عالية الجودة وباللهجة/الأسلوب نفسه، أضف كـ Kaggle Secrets (من Add-ons → Secrets):
`SUPABASE_URL` و `SUPABASE_SERVICE_ROLE_KEY`، ثم شغّل الخلية التالية. إن لم تُضفها، تجاوز هذه الخلية بأمان — بقية الدفتر يعمل بدونها.


In [ ]:
# SHAM_SECTION: sham_small_training
try:
    from data_acquisition import export_nova_knowledge_to_corpus
    own_corpus_path = "/kaggle/working/corpus/nova_own_knowledge.txt"
    n = export_nova_knowledge_to_corpus(own_corpus_path)
    if n > 0:
        corpus_files.append(own_corpus_path)
    print(f"تمت إضافة {n} فقرة حقيقية من معرفة شام الحالية إلى بيانات التدريب.")
except Exception as exc:
    print(f"تم تجاوز هذا المصدر الاختياري (طبيعي إن لم تُضف Kaggle Secrets الخاصة به): {exc}")


### 4) أداة تقسيم النص (Tokenizer)

**مهم جداً عند الاستئناف من جلسة سابقة:** إن كنت أضفت نتاج جلسة سابقة كـ Input (لاستئناف تدريب متوقف)، يجب استخدام **نفس أداة تقسيم النص** التي استُخدمت لتدريب تلك النقطة بالضبط — تدريب أداة جديدة الآن سينتج ربطاً مختلفاً بين الأرقام والكلمات، مما يفسد كل ما تعلّمه النموذج بصمت دون أي خطأ ظاهر. لهذا تبحث الخلية التالية أولاً عن أداة محفوظة من جلسة سابقة، ولا تُدرّب أداة جديدة إلا إن لم تجد شيئاً (أول تشغيل فعلي).


In [ ]:
# SHAM_SECTION: sham_small_training
from pathlib import Path
from text_tokenizer import train_text_tokenizer, ShamTextTokenizer
from model import TEXT_VOCAB_SIZE

# عند الاستئناف: نفس أداة تقسيم النص المحفوظة مع نقطة الحفظ المُستأنَفة بالضبط — أداة جديدة
# تنتج ربطاً مختلفاً بين الأرقام والكلمات فتفسد كل ما تعلّمه النموذج بصمت.
if LINEAGE:
    tokenizer = ShamTextTokenizer.load(str(LINEAGE["tokenizer"]))
    print(f"تم إعادة استخدام أداة تقسيم النص المحفوظة مع نقطة الحفظ (vocab={tokenizer.vocab_size}).")
else:
    assert corpus_files, "لا توجد ملفات نصية حقيقية — تحقق من نجاح خلية جمع البيانات أعلاه."
    tokenizer = train_text_tokenizer(corpus_files, vocab_size=TEXT_VOCAB_SIZE)
    print(f"تم تدريب أداة تقسيم نص حقيقية جديدة (vocab={tokenizer.vocab_size}) — أول تشغيل فعلي.")

tokenizer.save("/kaggle/working/sham_small_tokenizer.json")

### 5) بناء بيانات التدريب الفعلية (نافذات نصية بطول ثابت)

هذه الخطوة تحوّل كل النصوص المجمّعة إلى دفعات (batches) جاهزة للتدريب مباشرة.


In [ ]:
# SHAM_SECTION: sham_small_training
import torch
from dataset import TextSequenceDataset

SEQ_LEN = 1024

text_dataset = TextSequenceDataset(corpus_files, tokenizer, seq_len=SEQ_LEN)
print(f"عدد نوافذ التدريب الحقيقية الجاهزة: {len(text_dataset):,} (كل نافذة = {SEQ_LEN} رمزاً)")
assert len(text_dataset) >= 32, (
    "عدد نوافذ التدريب قليل جداً — كبّر MAX_DOCUMENTS في خلية جمع البيانات أعلاه وأعد التشغيل من هناك."
)


### 6) إعداد حجم النموذج

النموذج مبني بالكامل ليتوسع لاحقاً بلا فقدان أي تقدّم مُدرَّب (عبر `expand_model.py`) — لذلك نبدأ بحجم "بداية" آمن يتناسب مع ذاكرة معالج Kaggle المجاني (GPU مجاني عادة 16GB)، ثم نكبّره لاحقاً في جولات قادمة دون الحاجة لإعادة التدريب من الصفر.

- الحجم الافتراضي الكامل للمعمارية (~500 مليون معامل) متاح أيضاً أدناه كخيار — إن كان لديك GPU أقوى.


In [ ]:
# SHAM_SECTION: sham_small_training
from model import ShamSmallConfig, ShamSmall, TOTAL_VOCAB_SIZE

MODEL_SIZE = "starter"  # غيّرها إلى "full" إذا كان لديك GPU أقوى من الـ GPU المجاني القياسي

if MODEL_SIZE == "starter":
    # ~108 مليون معامل حقيقي — آمن على GPU مجاني قياسي (16GB) لأول تشغيل حقيقي
    model_cfg = ShamSmallConfig(
        vocab_size=TOTAL_VOCAB_SIZE, d_model=768, n_layers=12, n_heads=12, n_kv_heads=4,
        mlp_hidden=2048, max_seq_len=SEQ_LEN, use_gradient_checkpointing=True,
    )
else:
    # الحجم الكامل الافتراضي للمعمارية (~500 مليون معامل) — يحتاج GPU أقوى/بدفعات أصغر
    model_cfg = ShamSmallConfig(vocab_size=TOTAL_VOCAB_SIZE, max_seq_len=SEQ_LEN, use_gradient_checkpointing=True)

model = ShamSmall(model_cfg)
n_params = model.count_parameters()
print(f"تم بناء النموذج: {n_params:,} معامل حقيقي (حجم: {MODEL_SIZE}).")

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"سيُستخدم للتدريب: {device}" + ("  (تحذير: لا يوجد GPU — تأكد من تفعيله من Settings → Accelerator)" if device == "cpu" else ""))


### 7) استئناف تدريب سابق إن وُجد

إذا كانت هذه ليست أول مرة تشغّل فيها هذا الدفتر (جلسة Kaggle تُغلق تلقائياً بعد 9-12 ساعة)، أضف نتاج (Output) الجلسة السابقة كـ Input جديد لهذا الدفتر، وستكمل هذه الخلية من آخر نقطة حفظ تلقائياً. إن كانت هذه أول مرة، ستبدأ من الصفر تلقائياً بلا أي إجراء إضافي منك.


In [ ]:
# SHAM_SECTION: sham_small_training
from checkpoint import load_checkpoint

start_step = 0
resume_optimizer = None
if LINEAGE:
    model, start_step, _ = load_checkpoint(LINEAGE["checkpoint"], map_location=device)
    if LINEAGE["has_optimizer"]:
        from train import build_optimizer
        resume_optimizer = build_optimizer(model, lr=3e-4, weight_decay=0.1)
        load_checkpoint(LINEAGE["checkpoint"], map_location=device, load_optimizer_into=resume_optimizer)
    print(f"تم استئناف التدريب من {LINEAGE['dataset']}/{LINEAGE['checkpoint'].name} (الخطوة {start_step:,})")
else:
    print("لم يتم العثور على نقطة حفظ سابقة — سيبدأ التدريب من الصفر (هذا طبيعي في أول تشغيل).")

### 8) قياس السرعة الحقيقية قبل تحديد عدد الخطوات

بدل تخمين عدد الخطوات، نقيس فعلياً كم خطوة في الثانية يحققها معالج Kaggle المخصص لك الآن، ثم نحسب عدد خطوات واقعي يتناسب مع حد الجلسة (9–12 ساعة).


In [ ]:
# SHAM_SECTION: sham_small_training
import time
from train import TrainConfig, build_optimizer, build_lr_scheduler

# خطوات "إحماء" تُنفَّذ فعلياً لكن لا تُحتسب ضمن قياس السرعة -- أول خطوات
# حقيقية على أي جهاز (GPU أو CPU) تتضمن تكاليف لمرة واحدة فقط (حجز ذاكرة،
# لمس أول للصفحات، إحماء ذاكرة التخزين المؤقت) تجعل أي قياس على الخطوات
# الأولى مباشرة متشائماً أكثر من اللازم. حادثة حقيقية (ملاحظة المالكة،
# 2026-09-21): معايرة على الخطوات الأولى فقط أنتجت ميزانية خطوات متحفظة
# جداً، فأنهت الجلسة كل خطواتها المخصصة في ~ساعتين وتوقفت بشكل نظيف --
# بينما السرعة الحقيقية المستمرة كانت تسمح باستغلال أغلب MAX_TRAINING_HOURS.
# القياس بعد تجاوز الإحماء يعطي تقديراً واقعياً للسرعة المستمرة الفعلية.
CALIBRATION_WARMUP_STEPS = 5
CALIBRATION_STEPS = 30

calib_batches = [
    torch.stack([text_dataset[i] for i in range(b, b + 2)])
    for b in range(0, min(len(text_dataset) - 2, (CALIBRATION_WARMUP_STEPS + CALIBRATION_STEPS) * 2 * 4), 2)
][: (CALIBRATION_WARMUP_STEPS + CALIBRATION_STEPS) * 4]
assert len(calib_batches) > CALIBRATION_WARMUP_STEPS, "لا توجد بيانات كافية للقياس -- كبّر MAX_DOCUMENTS في خلية جمع البيانات."

model.to(device)
model.train()
_calib_optimizer = resume_optimizer or build_optimizer(model, lr=3e-4, weight_decay=0.1)

def _calib_step(batch):
    batch = batch.to(device)
    _, loss = model(batch, labels=batch)
    loss.backward()
    _calib_optimizer.step()
    _calib_optimizer.zero_grad()

for batch in calib_batches[:CALIBRATION_WARMUP_STEPS]:
    _calib_step(batch)

t0 = time.time()
steps_done = 0
for batch in calib_batches[CALIBRATION_WARMUP_STEPS:]:
    _calib_step(batch)
    steps_done += 1
    if steps_done >= CALIBRATION_STEPS:
        break
elapsed = time.time() - t0
steps_per_second = steps_done / elapsed

# قيمة ثابتة وآمنة تحت حد طول الجلسة الواحدة في Kaggle (لوحظ فعلياً عند
# ~12 ساعة قبل القتل القسري) -- وليست مرتبطة برصيدك الأسبوعي المتبقي، لأن
# هذه الخلية قد تعمل الآن بلا إشراف بشري (تشغيل مجدول تلقائي، انظر الخلية
# 11 أدناه)، فلا يوجد من يعدّلها يدوياً قبل كل جلسة. إن كان رصيدك الأسبوعي
# المتبقي أقل من هذه الساعات، ستنتهي الجلسة ببساطة بسبب نفاد الرصيد نفسه
# (وليس بسبب هذا الرقم) -- وهذا آمن تماماً بفضل نظام الاستئناف: الجلسة
# المجدولة القادمة (بعد تجدّد الرصيد الأسبوعي) تكمل من حيث توقفت تلقائياً.
# غيّرها يدوياً فقط إن كنت تراقب الجلسات بنفسك دون جدولة تلقائية.
MAX_TRAINING_HOURS = 8.5
realistic_steps_for_session = int(steps_per_second * MAX_TRAINING_HOURS * 3600 * 0.85)  # هامش أمان 15% إضافي

print(f"سرعة حقيقية مقاسة الآن (بعد تجاوز {CALIBRATION_WARMUP_STEPS} خطوات إحماء): {steps_per_second:.3f} خطوة/ثانية على {device}")
print(f"عدد خطوات واقعي يمكن إنجازه ضمن {MAX_TRAINING_HOURS} ساعة (بهامش أمان): {realistic_steps_for_session:,} خطوة")


### 9) التدريب الحقيقي

عدد الخطوات أدناه محسوب تلقائياً من القياس الحقيقي أعلاه — لا حاجة لتعديله يدوياً، لكن يمكنك تصغيره لو أردت تشغيلاً أقصر للتجربة أولاً (مثلاً بوضع `TOTAL_STEPS = 500`).

**حماية حقيقية إضافية:** حتى لو كان تقدير الخطوات أعلاه متفائلاً أكثر من اللازم، `max_wall_clock_seconds` أدناه يجعل التدريب **يوقف نفسه فعلياً** ويحفظ نقطة حفظ حقيقية بمجرد الاقتراب من `MAX_TRAINING_HOURS` — بدل انتظار Kaggle ليقتل الجلسة بالقوة بلا أي حفظ (كما حدث فعلياً من قبل).


In [ ]:
# SHAM_SECTION: sham_small_training
GRAD_ACCUM = 4   # دفعات صغيرة (4 نوافذ) لكل خطوة محسِّن
_streaming = getattr(text_dataset, "_streaming", None) is not None
if _streaming:
    # خط النص المتدفق (sham_text_stream.py): نوافذ جديدة بلا نهاية، لا شريحة ثابتة ولا تكرار. الجلسة تنتهي
    # بالساعة (MAX_TRAINING_HOURS)، ومعدل التعلّم يتناقص مع الساعة حتى حدّه الأدنى عند نهايتها.
    TOTAL_STEPS = 1_000_000
    print("خط نص متدفق: تدريب بلا حد خطوات حتى نهاية المدة المحددة — كل نافذة تُرى مرة واحدة فقط.")
else:
    # وحدات صحيحة: قياس السرعة أعلاه بخطوة = نافذتان، والخطوة الحقيقية = 16 نافذة
    TOTAL_STEPS = max(realistic_steps_for_session // 8, 200)
num_windows = len(text_dataset) - (len(text_dataset) % 4)
print(f"عدد نوافذ التدريب المتاحة: {num_windows:,} — الخطوات المخططة: {TOTAL_STEPS:,}")

def _batch_iterator():
    # تبني كل دفعة عند الحاجة فقط بدل تجهيز كل دفعات التدريب مسبقاً في
    # قائمة واحدة ضخمة (وتكرارها) — التجهيز المسبق كان يُبقي نسخة كاملة
    # إضافية من كل نوافذ التدريب في RAM طوال الجلسة، فوق النسخة التي
    # يحتفظ بها text_dataset نفسه أصلاً، وهو ما أسقط جلسة Colab فعلياً
    # بعد استنفاد كل RAM المتاح (حادثة حقيقية، 2026-09-15).
    while True:
        for b in range(0, num_windows, 4):
            yield torch.stack([text_dataset[i] for i in range(b, b + 4)])

import itertools
batches = _batch_iterator() if _streaming else itertools.islice(_batch_iterator(), TOTAL_STEPS * GRAD_ACCUM)

train_cfg = TrainConfig(
    seq_len=SEQ_LEN,
    batch_size=4,
    grad_accum_steps=GRAD_ACCUM,   # حجم دفعة فعلي = 16
    lr=3e-4,
    warmup_steps=300 if _streaming else max(50, TOTAL_STEPS // 100),
    total_steps=start_step + TOTAL_STEPS,
    checkpoint_dir="/kaggle/working/checkpoints",
    checkpoint_every=200,
    log_every=20,
    max_wall_clock_seconds=MAX_TRAINING_HOURS * 3600,
)

from train import train
loss_history = train(
    model, batches, train_cfg, device=device,
    start_step=start_step, resume_optimizer=resume_optimizer or _calib_optimizer,
)

print(f"\nانتهى التدريب على {len(loss_history):,} خطوة حقيقية.")
print(f"متوسط الخسارة (loss) في أول 10 خطوات: {sum(loss_history[:10]) / min(10, len(loss_history)):.4f}")
print(f"متوسط الخسارة (loss) في آخر 10 خطوات: {sum(loss_history[-10:]) / min(10, len(loss_history)):.4f}")
try:
    import sham_text_stream
    print(sham_text_stream.report())
except Exception:
    pass


### 10) حفظ النتيجة النهائية

آخر نقطة حفظ محفوظة أصلاً تلقائياً أثناء التدريب في `/kaggle/working/checkpoints/`. هذه الخلية تحفظ أيضاً نسخة أخيرة صريحة + أداة تقسيم النص، بحيث كل ما تحتاجه لاستئناف التدريب أو لتشغيل النموذج عبر `serve.py` موجود في نتاج هذه الجلسة (Output) تلقائياً — لا حاجة لتنزيل أي شيء يدوياً؛ بعد انتهاء تشغيل الدفتر اضغط **Save Version** ليصبح هذا الناتج قابلاً لإضافته كـ Input لجلسة تدريب أو خدمة قادمة.


In [ ]:
# SHAM_SECTION: sham_small_training
from checkpoint import save_checkpoint

final_step = start_step + len(loss_history)
save_checkpoint("/kaggle/working/checkpoints/final.pt", model, final_step)
print(f"تم حفظ النقطة النهائية عند الخطوة {final_step:,} في /kaggle/working/checkpoints/final.pt")
print("أداة تقسيم النص محفوظة في /kaggle/working/sham_small_tokenizer.json")
print("\nلا تنسَ: اضغط الآن Save Version أعلى الصفحة حتى يُحفظ كل هذا كـ Output دائم لهذه الجلسة.")


### 11) نشر نقطة الحفظ تلقائياً كنسخة جديدة على Kaggle Dataset — لسلسلة تدريب بلا تدخل يدوي

**هذه الخلية هي ما يجعل التدريب يُكمل نفسه بنفسه عبر عدة جلسات، دون أن تفتح الهاتف أو تنشئ Dataset يدوياً كل مرة.**

آلية العمل:
1. تنشر هذه الخلية آخر نقطة حفظ + أداة تقسيم النص كـ **نسخة جديدة (version)** من نفس الـ Dataset (`sham-checkpoint`) — عبر Kaggle API مباشرة من داخل الكود؛ الخلية تُنشئه تلقائياً بنفسها إن لم يكن موجوداً بعد.
2. إن كان هذا الدفتر **مُجدولاً للعمل تلقائياً** (Schedule a notebook to run)، فالجلسة القادمة ستجد هذا الـ Dataset بآخر نسخة منه مرفقاً تلقائياً كـ Input (بشرط عدم تثبيته على إصدار معيّن)، فتلتقط نقطة الحفظ الجديدة تلقائياً بلا أي تدخل.

**إعداد لمرة واحدة فقط قبل استخدام هذه الخلية:**
1. اذهب إلى https://www.kaggle.com/settings → قسم **API** → **Create New Token** — سيُنزَّل ملف `kaggle.json` يحوي `username` و `key`.
2. في هذا الدفتر: **Add-ons → Secrets → Add a new secret** مرتين:
   - الاسم `KAGGLE_USERNAME` والقيمة هي `username` من الملف.
   - الاسم `KAGGLE_KEY` والقيمة هي `key` من الملف.

**بعد إعداد هذا مرة واحدة:** من القائمة الجانبية اضغط **"Schedule a notebook to run"** واختر تكراراً (مثلاً كل يوم) — ومن الآن فصاعداً لن تحتاج فتح Kaggle يدوياً إطلاقاً؛ فقط راقب تقدّم الخسارة (loss) بين حين وآخر.

In [ ]:
# SHAM_SECTION: sham_small_training
import json as _json
import os as _os
import subprocess as _subprocess
import shutil as _shutil
from pathlib import Path as _Path
from kaggle_secrets import UserSecretsClient as _UserSecretsClient

_subprocess.run(["pip", "install", "-q", "-U", "kaggle"], check=False)

KAGGLE_USERNAME = _UserSecretsClient().get_secret("KAGGLE_USERNAME")
KAGGLE_KEY = _UserSecretsClient().get_secret("KAGGLE_KEY")
# "sham-checkpoint": renamed off the old "nova-small-checkpoint" name left
# over from before the product was renamed to Sham (owner-caught,
# 2026-09-20) -- and off the short-lived "-v2" name too, before that name
# ever had a real version published under it.
DATASET_SLUG = f"{KAGGLE_USERNAME}/sham-checkpoint"

_os.environ["KAGGLE_USERNAME"] = KAGGLE_USERNAME
_os.environ["KAGGLE_KEY"] = KAGGLE_KEY

upload_dir = _Path("/kaggle/working/for_dataset_upload")
if upload_dir.exists():
    _shutil.rmtree(upload_dir)
(upload_dir / "checkpoints").mkdir(parents=True)
for ckpt in _Path("/kaggle/working/checkpoints").glob("*.pt"):
    _shutil.copy2(ckpt, upload_dir / "checkpoints" / ckpt.name)
_shutil.copy2("/kaggle/working/sham_small_tokenizer.json", upload_dir / "sham_small_tokenizer.json")
for _extra in _Path("/kaggle/working/checkpoints").glob("*.json"):  # موضع القراءة من ويكيبيديا
    _shutil.copy2(_extra, upload_dir / _extra.name)

metadata = {"title": "sham-checkpoint", "id": DATASET_SLUG, "licenses": [{"name": "unknown"}]}
(upload_dir / "dataset-metadata.json").write_text(_json.dumps(metadata))

# Deterministic existence check -- ask Kaggle directly whether this
# dataset is already in the account's own list, instead of guessing from
# an error message's exact wording (not a stable, documented contract to
# parse -- the earlier version of this cell relied on spotting "404" in
# the output, which is exactly the kind of fragility to avoid).
from sham_inputs import dataset_exists as _dataset_exists_fn   # reads every page of the list (2026-10-10: page 1 only hid older datasets)
_dataset_exists = _dataset_exists_fn(DATASET_SLUG)

# "-r zip" (not "skip"): "-r skip" is Kaggle CLI's documented default for
# subdirectories -- it silently IGNORES them rather than uploading them.
# checkpoints/ is a subdirectory, so every earlier version published under
# "-r skip" never actually contained the .pt files, only the flat
# tokenizer.json alongside it -- despite this cell printing success every
# time (owner-caught, 2026-09-20). "-r zip" actually uploads the
# subdirectory, as a .zip archive.
if _dataset_exists:
    result = _subprocess.run(
        ["kaggle", "datasets", "version", "-p", str(upload_dir), "-m", f"auto-update at step {final_step:,}", "-r", "zip"],
        capture_output=True, text=True,
    )
else:
    result = _subprocess.run(
        ["kaggle", "datasets", "create", "-p", str(upload_dir), "-r", "zip"],
        capture_output=True, text=True,
    )
_combined = (result.stdout or "") + (result.stderr or "")

if result.returncode == 0 and "error" not in _combined.lower():
    verb = "نُشرت نسخة جديدة إلى" if _dataset_exists else "تم إنشاء"
    print(f"{verb} {DATASET_SLUG} عند الخطوة {final_step:,} — الجلسة القادمة (المجدولة) ستلتقطها تلقائياً.")
elif "incompatible" in _combined.lower():
    # Should not happen on a name this fresh, but kept as a clear,
    # actionable message rather than a silent dead end.
    print(
        "تنبيه: هذا الـDataset أُنشئ تحت وضع رفع غير متوافق. لا يوجد إصلاح عبر الكود لهذا الاسم تحديداً -- "
        "غيّري DATASET_SLUG أعلاه لاسم لم يستخدمه كاجل من قبل (مثلاً sham-checkpoint-v2) وأعيدي التشغيل. "
        "نقطة الحفظ نفسها آمنة حالياً في Output هذه الجلسة بغض النظر."
    )
    print(_combined)
else:
    print("تنبيه: فشل نشر النسخة الجديدة تلقائياً — نقطة الحفظ نفسها لا تزال محفوظة بأمان في Output هذه الجلسة. "
          "تحقق من صحة KAGGLE_USERNAME / KAGGLE_KEY.")
    print(_combined)

### 12) إرسال تقرير الجلسة للمالك عبر تيليجرام (اختياري)

نفس فكرة تقرير نوفا الأسبوعي التلقائي على تيليجرام، مطبّقة الآن على جلسات شام الحقيقية أيضاً — راجع `telegram_report.py`. يتخطى الإرسال بأمان تماماً إن لم يُضبط `TELEGRAM_BOT_TOKEN`/`TELEGRAM_CHAT_ID` كأسرار (Secrets) لهذا الدفتر.

In [ ]:
# SHAM_SECTION: sham_small_training
try:
    from kaggle_secrets import UserSecretsClient
    TELEGRAM_BOT_TOKEN = UserSecretsClient().get_secret("TELEGRAM_BOT_TOKEN")
    TELEGRAM_CHAT_ID = UserSecretsClient().get_secret("TELEGRAM_CHAT_ID")
except Exception:
    # Both secrets are OPTIONAL -- a real, one-time setup step (owner
    # request, 2026-09-21, comparing Nova's automatic Telegram reports
    # against Sham's total silence between manual Kaggle-log checks).
    # Add-ons -> Secrets -> add TELEGRAM_BOT_TOKEN (an existing bot's
    # token) and TELEGRAM_CHAT_ID (the owner's own Telegram numeric id,
    # e.g. the SUPER_ADMIN_TELEGRAM_ID already used elsewhere in this
    # project) to receive this. Not configured -> report is skipped,
    # never blocks or fails the session itself.
    TELEGRAM_BOT_TOKEN, TELEGRAM_CHAT_ID = None, None

from telegram_report import format_training_report, send_telegram_message

_published_slug = DATASET_SLUG if (result.returncode == 0 and "error" not in _combined.lower()) else None
_report_text = format_training_report("المسار الرئيسي (GPU)", final_step, loss_history, _published_slug)
print(_report_text)
send_telegram_message(TELEGRAM_BOT_TOKEN, TELEGRAM_CHAT_ID, _report_text)


---
# القسم: المرحلة الثانية — الصورة والصوت مع النص
`sham_small_training_stage2_multimodal` — خلاياه أدناه كما هي في دفتره الأصلي.

In [ ]:
# SHAM_SECTION: sham_small_training_stage2_multimodal
get_ipython().run_line_magic("reset", "-f")   # قسم جديد = دفتر جديد: لا متغيرات من قسم سابق
import gc

gc.collect()
try:
    import torch
    torch.cuda.empty_cache()   # يحرّر ذاكرة GPU بين قسمين في جلسة واحدة
except Exception:
    pass


# شام — المرحلة الثانية: إضافة الصورة والصوت (Vision + Audio)

هذا الدفتر **يكمل** تدريب النموذج الذي أنتجه دفتر المرحلة الأولى (النص) — لا يبدأ من الصفر، ولا يفقد أي شيء تعلّمه النموذج من النص. يضيف عليه فقط قدرتين حقيقيتين جديدتين:
- **الصورة**: توليد صور من نص (Generation) + وصف/فهم صورة معطاة (Understanding) — بالاتجاهين معاً من نفس البيانات.
- **الصوت**: تحويل نص لصوت (Generation) + تفريغ/فهم صوت معطى (Understanding) — بالاتجاهين معاً أيضاً.

**نطاق حقيقي يجب معرفته بصراحة:** بيانات الصور المتاحة بكثرة وجاهزة للسحب الآن هي بتعليقات **إنجليزية** (Flickr30k) — لا يوجد مصدر عربي مكافئ بنفس الحجم متاح فوراً. هذا لا يضر الآلية نفسها (تعلّم ربط البكسل برموز، وربطها بنص التعليق) لأنها مستقلة عن اللغة تماماً، لكن يعني أن وصف الصور بالعربية الفصيحة يحتاج بيانات عربية لاحقاً لتحسينه أكثر. الصوت بالمقابل عربي حقيقي بالكامل (Common Voice العربية).

## قبل الضغط على "Save Version → Save & Run All" — خطوتان:

1. **لا حاجة لأي «Add Input»:** الدفتر يجلب بنفسه عبر Kaggle API آخر نسخة من: نقطة حفظ المرحلة الأولى (`sham-checkpoint`)، وأداتي ترميز الصورة والصوت (`sham-image-tokenizer-checkpoint`، `sham-audio-tokenizer-checkpoint`)، وناتجه هو السابق (`sham-multimodal-checkpoint`) إن وُجد فيكمل منه بدل البدء من المرحلة الأولى من جديد.
2. **الأسرار:** `GITHUB_TOKEN`، `KAGGLE_USERNAME`، `KAGGLE_KEY` — وفعّلي الإنترنت واختاري GPU من Settings.

في النهاية ينشر النتيجة إلى `sham-multimodal-checkpoint` (النموذج + أداة تقسيم النص + أداتا الترميز + سجلات البيانات) — منها يُشغَّل خادم شام ودفتر تجربة البوت.

**مهم:** استخدم **Save Version → Save & Run All (Commit)** مباشرة من البداية لهذا الدفتر (وليس التشغيل التفاعلي) — تعلمنا من المرحلة الأولى أن هذا هو الأسلوب الآمن الوحيد لتشغيل يستغرق ساعات دون قلق من إغلاق الهاتف أو انقطاع الاتصال.

> **CPU أو GPU — اختيارك (2026-09-24):** الدفتر يعمل على الاثنين. إن ضبطتِه على GPU ونفد رصيد GPU المجاني، يُعيده منسّق الاستئناف تلقائياً على المعالج العادي (CPU) ويكمل من نفس النقطة، ثم يرجعه إلى GPU حين يتجدد الرصيد. وإن ضبطتِه على CPU فلا يُنقل إلى GPU أبداً.

### 1) سحب الكود الحقيقي من GitHub مباشرة (نفس أسلوب المرحلة الأولى)

In [ ]:
# SHAM_SECTION: sham_small_training_stage2_multimodal
import os
import sys
import subprocess
from kaggle_secrets import UserSecretsClient

GITHUB_TOKEN = UserSecretsClient().get_secret("GITHUB_TOKEN")
REPO_URL = f"https://{GITHUB_TOKEN}@github.com/jonsnow-org/Ttbik.git"
BRANCH = "claude/free-services-marketplace-h6rwk2"
CLONE_DIR = "/kaggle/working/Ttbik"

if not os.path.exists(CLONE_DIR):
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, CLONE_DIR], check=True)
    print("تم سحب الكود الحقيقي من مستودع GitHub بنجاح.")
else:
    subprocess.run(["git", "-C", CLONE_DIR, "pull"], check=True)
    print("الكود موجود بالفعل في هذه الجلسة — تم سحب أي تحديثات جديدة عليه.")

# أمان: git يحفظ رابط الاستنساخ (بما فيه GITHUB_TOKEN) حرفياً داخل
# .git/config -- وهذا المجلد يبقى ضمن نتاج (Output) هذه الجلسة، الذي قد
# يُستخدَم لاحقاً كمُدخَل (Notebook Output) لجلسة أخرى، أو يُشارَك بأي شكل.
# نزع التوكن من الرابط المحفوظ فور نجاح الاستنساخ يمنع تسربه عبر هذا
# المسار تماماً (ثغرة حقيقية اكتشفتها المالكة، 2026-09-21).
subprocess.run(["git", "-C", CLONE_DIR, "remote", "set-url", "origin", "https://github.com/jonsnow-org/Ttbik.git"], check=True)

CODE_DIR = os.path.join(CLONE_DIR, "ai-system", "colab", "sham_small")
assert os.path.exists(os.path.join(CODE_DIR, "model.py")), f"لم يتم العثور على model.py داخل {CODE_DIR}"
sys.path.insert(0, CODE_DIR)
print("كود ShamSmall الحقيقي جاهز في:", CODE_DIR)


### 2) تثبيت المكتبات الإضافية

In [ ]:
# SHAM_SECTION: sham_small_training_stage2_multimodal
try:
    import tokenizers
    print(f"مكتبة tokenizers متوفرة مسبقاً (نسخة {tokenizers.__version__}).")
except ImportError:
    subprocess.run(["pip", "install", "-q", "tokenizers"], check=True)
    print("تم تثبيت مكتبة tokenizers.")

try:
    import soundfile
    print("مكتبة soundfile متوفرة مسبقاً.")
except ImportError:
    subprocess.run(["pip", "install", "-q", "soundfile"], check=True)
    print("تم تثبيت مكتبة soundfile.")


### 3) تحميل نقطة الحفظ وأداة تقسيم النص من المرحلة الأولى

هذه الخلية تبحث عن نتاج دفتر المرحلة الأولى الذي أضفته كـ Input (الخطوة 1 أعلاه) — إن لم تجدها، ستتوقف برسالة واضحة تخبرك بالضبط ما الناقص.


In [ ]:
# SHAM_SECTION: sham_small_training_stage2_multimodal
from pathlib import Path
from checkpoint import load_checkpoint
from text_tokenizer import ShamTextTokenizer
from sham_inputs import fetch_dataset, resume_text_lineage

device = "cuda" if __import__("torch").cuda.is_available() else "cpu"
print(f"سيُستخدم للتدريب: {device}" + ("  (تحذير: لا يوجد GPU — فعّله من Settings → Accelerator)" if device == "cpu" else ""))

# كل المدخلات تُجلب تلقائياً (مرفقة، أو آخر نسخة عبر Kaggle API).
for _name in ("sham-image-tokenizer-checkpoint", "sham-audio-tokenizer-checkpoint"):
    fetch_dataset(_name)

# ناتج هذه المرحلة نفسها إن نُشر سابقاً: نكمل منه (مع أداة تقسيم النص وأداتي الترميز المحفوظة
# معه نفسها)، وإلا نبدأ من نقطة حفظ المرحلة الأولى.
STAGE2_OWN = fetch_dataset("sham-multimodal-checkpoint")
_own_ckpt = sorted(STAGE2_OWN.rglob("final_multimodal.pt")) if STAGE2_OWN else []
_own_tok = sorted(STAGE2_OWN.rglob("sham_small_tokenizer.json")) if STAGE2_OWN else []
if _own_ckpt and _own_tok:
    stage1_checkpoint_path, _tokenizer_path = _own_ckpt[0], _own_tok[0]
    print("استئناف المرحلة الثانية من ناتجها السابق.")
else:
    STAGE2_OWN = None
    LINEAGE = resume_text_lineage(
        "sham-checkpoint", forks=["nova-small-checkpoint"], tokenizer_pattern="*tokenizer*.json",
    )
    assert LINEAGE, (
        "لم يُعثر على نقطة حفظ المرحلة الأولى (sham-checkpoint) — شغّلي دفتر المرحلة الأولى أولاً حتى ينشرها."
    )
    stage1_checkpoint_path, _tokenizer_path = LINEAGE["checkpoint"], LINEAGE["tokenizer"]

model, start_step, _ = load_checkpoint(stage1_checkpoint_path, map_location=device)
print(f"تم تحميل نقطة الحفظ: {stage1_checkpoint_path} (الخطوة {start_step:,})")
tokenizer = ShamTextTokenizer.load(str(_tokenizer_path))
tokenizer.save("/kaggle/working/sham_small_tokenizer.json")
print(f"تم تحميل أداة تقسيم النص المحفوظة معها (vocab={tokenizer.vocab_size}).")

### 4) جمع بيانات صورة وصوت حقيقية

- **الصور**: نحاول أولاً مصدراً عربياً حقيقياً حديثاً (`Misraj/Arabic-Image-Captioning_100M` — 100 مليون زوج صورة+تعليق عربي حقيقي)، والخلية أدناه تتراجع تلقائياً لـ Flickr30k (تعليقات إنجليزية) إن فشل المصدر العربي لأي سبب واقعي (تغيّر اسم عمود، تغيّر الرابط، إلخ) — **هذا تحقّق حقيقي وقت التشغيل، وليس افتراضاً**: لم أتمكن من فتح صفحة هذا المصدر مباشرة للتأكد من أسماء أعمدته بالضبط (huggingface.co محجوب من بيئة التطوير التي بنيت منها هذا الكود)، فالكود يفحص الأعمدة الحقيقية فور وصول أول عنصر حقيقي، ويخبرك فوراً إن كانت مختلفة عمّا توقعته بدل الفشل الصامت.
- **الصوت**: Common Voice العربية (صوت وتفريغ نصي عربي حقيقي وموثوق)، عبر streaming.

ابدأ بعدد معقول (كما في المرحلة الأولى) للتأكد أن كل شيء يعمل، ثم كبّره في تشغيل لاحق.


In [ ]:
# SHAM_SECTION: sham_small_training_stage2_multimodal
from sham_data_sources import Ledger, collect

MAX_IMAGES = 4_000
MAX_AUDIO_SAMPLES = 3_000
# الصور تُجمع بدقة أداة ترميز الصورة نفسها (256) — كانت 64 ثم تُكبَّر فيتعلّم النموذج صوراً ضبابية.
from image_tokenizer import ImageTokenizerConfig as _ITC
COLLECT_IMAGE_SIZE = _ITC().image_size

# كل تشغيل يأخذ بيانات جديدة لم يرها النموذج من قبل: موضع استئناف لكل مصدر + بصمة لكل
# عيّنة، في سجلّين خاصين بالمرحلة الثانية (stage2_image / stage2_audio) يُحفظان في
# /kaggle/working/checkpoints — أضف نتاج هذا الدفتر كمدخل للتشغيل التالي ليكمل بعده.
# (سابقاً: نفس أول 3000 صورة في كل تشغيل، ومصدر Common Voice صار فارغاً على Hugging Face.)
image_ledger = Ledger.load("image", name="stage2_image")
audio_ledger = Ledger.load("audio", name="stage2_audio")
# أزواج ثابتة لقياس ربط النص بالصورة/الصوت قبل الجلسة وبعدها — تُستبعد من التدريب.
from sham_media_link import collect_fixed_pairs
FIXED_PAIRS = collect_fixed_pairs(("image", "audio"), {"image": image_ledger, "audio": audio_ledger}, image_size=COLLECT_IMAGE_SIZE)
image_manifest, image_collect_stats = collect("image", "/kaggle/working/corpus/images", MAX_IMAGES, image_ledger, image_size=COLLECT_IMAGE_SIZE)
audio_manifest, audio_collect_stats = collect("audio", "/kaggle/working/corpus/audio", MAX_AUDIO_SAMPLES, audio_ledger)
# كل المصادر معاً (قرار 2026-10-11): ما جمعته هذه المرحلة مباشرة + عيّنة من دفعات «مصنع الوسائط» (استوديو Lightning)
import sham_media_shards
image_manifest = sham_media_shards.with_factory("image", image_manifest, "/kaggle/working/corpus/image_factory", k=1, limit=4_000) or image_manifest
audio_manifest = sham_media_shards.with_factory("audio", audio_manifest, "/kaggle/working/corpus/audio_factory", k=1, limit=3_000) or audio_manifest
Path("/kaggle/working/checkpoints").mkdir(parents=True, exist_ok=True)
image_ledger.save("/kaggle/working/checkpoints")
audio_ledger.save("/kaggle/working/checkpoints")
assert image_collect_stats.written + audio_collect_stats.written > 0, (
    "لا توجد بيانات جديدة غير مكررة في أي مصدر — المرحلة الثانية رأت كل البيانات المتاحة؛ أضف مصادر جديدة إلى SOURCES."
)
print(f"جاهز: {image_manifest}")
print(f"جاهز: {audio_manifest}")


### 5) تدريب أداتي ترميز الصورة والصوت (VQ-VAE) على البيانات الحقيقية

هاتان الأداتان تحوّلان صورة/مقطع صوت حقيقياً إلى "رموز" (tokens) يفهمها النموذج الرئيسي — يجب تدريبهما أولاً على بيانات حقيقية قبل استخدامهما (تدريبهما بأوزان عشوائية ينتج ضجيجاً بلا معنى).

حجم أداة ترميز الصورة هنا هو حجم "بداية" آمن (صور 64×64) يتناسب مع GPU المجاني — قابل للتكبير لاحقاً بنفس فلسفة النموذج الرئيسي.


In [ ]:
# SHAM_SECTION: sham_small_training_stage2_multimodal
import torch
from PIL import Image
import json as _json
from image_tokenizer import ImageTokenizer, ImageTokenizerConfig
from train_image_tokenizer import train_vqvae as train_image_vqvae, save_tokenizer_checkpoint as save_image_tokenizer
from train_image_tokenizer import load_tokenizer_checkpoint as load_image_tokenizer

# ربط تلقائي بشام: إن كانت أداة ترميز الصورة مدرّبة مسبقاً في مسارها المستقل (CPU)
# ومُضافة كمُدخل (Add Input) لهذا الدفتر، نستخدمها كما هي بدل تدريب أداة
# جديدة صغيرة من الصفر — لا حاجة لمعرفة اسم المجموعة أو مسارها، البحث تلقائي.
# يقبل المسارين تلقائياً بلا تعارض: أي مجموعة مُضافة تحوي image_tokenizer.pt
# (مسار الترميز المستقل، أو مجموعة مدمجة) — يُختار الأكثر تدريباً والمتوافق
# مع النموذج (tokenizer_select.py)، وإن لم يوجد شيء صالح يُدرَّب هنا من جديد.
from tokenizer_select import select_pretrained_tokenizer
print("البحث عن أداة ترميز جاهزة (image):")
# عند استئناف ناتج هذه المرحلة: نفس أداة الترميز التي تدرّب معها النموذج أولاً (أداة أحدث
# تعطي رموزاً مختلفة لنفس الصورة/الصوت)، وإلا الأكثر تدريباً بين المدخلات.
_picked_image = (select_pretrained_tokenizer("image", STAGE2_OWN) if STAGE2_OWN else None) or select_pretrained_tokenizer("image")
_pretrained_image = [_picked_image[2]] if _picked_image else []
if _picked_image:
    image_tokenizer, _image_step = _picked_image[0], _picked_image[1]
    image_tokenizer_cfg = image_tokenizer.cfg
    print(f"✅ استُخدمت أداة ترميز الصورة المدرّبة مسبقاً: {_pretrained_image[-1]} (خطوة {_image_step})")
    save_image_tokenizer("/kaggle/working/image_tokenizer.pt", image_tokenizer, step=_image_step)
else:
    print("لم يُعثر على أداة ترميز الصورة مدرّبة ضمن المدخلات — تدريب أداة جديدة هنا.")


    image_tokenizer_cfg = ImageTokenizerConfig(
        image_size=64, base_channels=32, channel_multipliers=(1, 2, 4), code_dim=64,  # حجم بداية آمن
        # num_codes يبقى ثابتاً على القيمة المعمارية الحقيقية (8192) — لا يُغيَّر أبداً، فالنموذج الرئيسي يحجز لها هذا المدى بالضبط.
    )
    image_tokenizer = ImageTokenizer(image_tokenizer_cfg)
    print(f"عدد رموز كل صورة: {image_tokenizer_cfg.tokens_per_image} (شبكة {image_tokenizer_cfg.latent_grid_size}x{image_tokenizer_cfg.latent_grid_size})")

    # تحميل الصور الحقيقية إلى مصفوفة واحدة في الذاكرة (مناسب لبضعة آلاف صورة بحجم 64x64)
    image_root = Path(image_manifest).parent
    images_list = []
    with open(image_manifest, encoding="utf-8") as f:
        for line in f:
            record = _json.loads(line)
            img = Image.open(image_root / record["image"]).convert("RGB").resize((image_tokenizer_cfg.image_size, image_tokenizer_cfg.image_size))
            tensor = torch.tensor(list(img.getdata()), dtype=torch.float32).view(image_tokenizer_cfg.image_size, image_tokenizer_cfg.image_size, 3)
            images_list.append(tensor.permute(2, 0, 1) / 127.5 - 1.0)
    real_images = torch.stack(images_list, dim=0)
    print(f"عدد الصور الحقيقية المحمّلة للتدريب: {real_images.shape[0]:,}")

    image_vqvae_stats = train_image_vqvae(image_tokenizer, real_images, num_epochs=10, batch_size=32, device=device, log_every=2)
    print(f"آخر خسارة إعادة بناء+VQ حقيقية: {image_vqvae_stats.epoch_losses[-1]:.4f} | "
          f"استخدام القاموس (codebook): {image_vqvae_stats.final_codebook_usage}/{image_vqvae_stats.codebook_size}")

    save_image_tokenizer("/kaggle/working/image_tokenizer.pt", image_tokenizer, step=len(image_vqvae_stats.epoch_losses))
    print("تم حفظ أداة ترميز الصورة الحقيقية المدرَّبة.")

# الترميز داخل MultimodalCollator يجري على CPU (الصور/المقاطع تُحمَّل على CPU)،
# ثم تُنقل الدفعات الجاهزة إلى GPU للنموذج الرئيسي — لذا تبقى أداة الترميز على
# CPU وفي وضع التقييم (مجمّدة) في الحالتين: المدرّبة مسبقاً والمدرّبة هنا.
image_tokenizer = image_tokenizer.to("cpu").train(False)

In [ ]:
# SHAM_SECTION: sham_small_training_stage2_multimodal
import soundfile as sf
from mel_spectrogram import waveform_to_mel_segments, waveform_to_mel_spectrogram
from audio_tokenizer import AudioTokenizer, AudioTokenizerConfig
from train_audio_tokenizer import train_vqvae as train_audio_vqvae, save_tokenizer_checkpoint as save_audio_tokenizer
from train_audio_tokenizer import load_tokenizer_checkpoint as load_audio_tokenizer

# ربط تلقائي بشام: إن كانت أداة ترميز الصوت مدرّبة مسبقاً في مسارها المستقل (CPU)
# ومُضافة كمُدخل (Add Input) لهذا الدفتر، نستخدمها كما هي بدل تدريب أداة
# جديدة صغيرة من الصفر — لا حاجة لمعرفة اسم المجموعة أو مسارها، البحث تلقائي.
# يقبل المسارين تلقائياً بلا تعارض: أي مجموعة مُضافة تحوي audio_tokenizer.pt
# (مسار الترميز المستقل، أو مجموعة مدمجة) — يُختار الأكثر تدريباً والمتوافق
# مع النموذج (tokenizer_select.py)، وإن لم يوجد شيء صالح يُدرَّب هنا من جديد.
from tokenizer_select import select_pretrained_tokenizer
print("البحث عن أداة ترميز جاهزة (audio):")
# عند استئناف ناتج هذه المرحلة: نفس أداة الترميز التي تدرّب معها النموذج أولاً (أداة أحدث
# تعطي رموزاً مختلفة لنفس الصورة/الصوت)، وإلا الأكثر تدريباً بين المدخلات.
_picked_audio = (select_pretrained_tokenizer("audio", STAGE2_OWN) if STAGE2_OWN else None) or select_pretrained_tokenizer("audio")
_pretrained_audio = [_picked_audio[2]] if _picked_audio else []
if _picked_audio:
    audio_tokenizer, _audio_step = _picked_audio[0], _picked_audio[1]
    audio_tokenizer_cfg = audio_tokenizer.cfg
    print(f"✅ استُخدمت أداة ترميز الصوت المدرّبة مسبقاً: {_pretrained_audio[-1]} (خطوة {_audio_step})")
    save_audio_tokenizer("/kaggle/working/audio_tokenizer.pt", audio_tokenizer, step=_audio_step)
else:
    print("لم يُعثر على أداة ترميز الصوت مدرّبة ضمن المدخلات — تدريب أداة جديدة هنا.")

    audio_tokenizer_cfg = AudioTokenizerConfig()  # الحجم الافتراضي مناسب هنا (num_codes يطابق AUDIO_VOCAB_SIZE أصلاً)
    audio_tokenizer = AudioTokenizer(audio_tokenizer_cfg)
    print(f"عدد رموز كل مقطع صوتي: {audio_tokenizer_cfg.tokens_per_segment}")

    audio_root = Path(audio_manifest).parent
    mels_list = []
    with open(audio_manifest, encoding="utf-8") as f:
        for line in f:
            record = _json.loads(line)
            waveform, sample_rate = sf.read(str(audio_root / record["audio"]), dtype="float32", always_2d=False)
            if waveform.ndim > 1:
                waveform = waveform.mean(axis=1)
            # كل مقاطع المقطع الصوتي (حتى 6)، لا اقتطاع 2.56 ثانية واحدة من منتصفه (إصلاح 2026-10-11)
            mels_list.extend(waveform_to_mel_segments(
                torch.from_numpy(waveform), sample_rate, audio_tokenizer_cfg.n_mels, audio_tokenizer_cfg.segment_frames, 6
            ))
    real_mels = torch.stack(mels_list, dim=0)
    print(f"عدد المقاطع الصوتية الحقيقية المحمّلة للتدريب: {real_mels.shape[0]:,}")

    audio_vqvae_stats = train_audio_vqvae(audio_tokenizer, real_mels, num_epochs=10, batch_size=32, device=device, log_every=2)
    print(f"آخر خسارة إعادة بناء+VQ حقيقية: {audio_vqvae_stats.epoch_losses[-1]:.4f} | "
          f"استخدام القاموس (codebook): {audio_vqvae_stats.final_codebook_usage}/{audio_vqvae_stats.codebook_size}")

    save_audio_tokenizer("/kaggle/working/audio_tokenizer.pt", audio_tokenizer, step=len(audio_vqvae_stats.epoch_losses))
    print("تم حفظ أداة ترميز الصوت الحقيقية المدرَّبة.")

# الترميز داخل MultimodalCollator يجري على CPU (الصور/المقاطع تُحمَّل على CPU)،
# ثم تُنقل الدفعات الجاهزة إلى GPU للنموذج الرئيسي — لذا تبقى أداة الترميز على
# CPU وفي وضع التقييم (مجمّدة) في الحالتين: المدرّبة مسبقاً والمدرّبة هنا.
audio_tokenizer = audio_tokenizer.to("cpu").train(False)

# تأكيد صريح: نفس أداتي الترميز المحفوظتين مع نقطة هذه المرحلة (لا إعادة تهيئة).
from sham_media_link import confirm_saved_tokenizers
confirm_saved_tokenizers(image_tokenizer, audio_tokenizer, STAGE2_OWN)

### 6) بناء دفعات تدريب حقيقية متعددة الوسائط (نص + صورة + صوت معاً)

كل دفعة هنا زوج حقيقي (input_ids, labels) — وليس نصاً عادياً — لأن الصور/الصوت تحتاج حشواً (padding)، تماماً كما تحقق ذلك في الإصلاح الذي أُجري على `train.py` مسبقاً.


In [ ]:
# SHAM_SECTION: sham_small_training_stage2_multimodal
from dataset import ImageCaptionDataset, AudioTranscriptDataset, MultimodalCollator, AudioMultimodalCollator

image_dataset = ImageCaptionDataset(image_manifest, image_size=image_tokenizer_cfg.image_size)
audio_dataset = AudioTranscriptDataset(audio_manifest, n_mels=audio_tokenizer_cfg.n_mels, segment_frames=audio_tokenizer_cfg.segment_frames, max_segments=4)
print(f"عدد أزواج (صورة، تعليق): {len(image_dataset)}")
print(f"عدد أزواج (صوت، نص): {len(audio_dataset)}")
from sham_media_link import data_report
data_report("الصورة", image_collect_stats, image_dataset)
data_report("الصوت", audio_collect_stats, audio_dataset)

image_collator = MultimodalCollator(tokenizer, image_tokenizer, both_directions=True)
audio_collator = AudioMultimodalCollator(tokenizer, audio_tokenizer, both_directions=True)

IMAGE_BATCH_SIZE = 4
AUDIO_BATCH_SIZE = 4

multimodal_batches = []
for start in range(0, len(image_dataset) - (len(image_dataset) % IMAGE_BATCH_SIZE), IMAGE_BATCH_SIZE):
    batch = [image_dataset[i] for i in range(start, start + IMAGE_BATCH_SIZE)]
    multimodal_batches.append(image_collator(batch))
for start in range(0, len(audio_dataset) - (len(audio_dataset) % AUDIO_BATCH_SIZE), AUDIO_BATCH_SIZE):
    batch = [audio_dataset[i] for i in range(start, start + AUDIO_BATCH_SIZE)]
    multimodal_batches.append(audio_collator(batch))

import random
random.Random(0).shuffle(multimodal_batches)  # لا نريد كل الصور أولاً ثم كل الصوت — نخلطهما فعلياً
print(f"عدد دفعات (نص+صورة) و(نص+صوت) الحقيقية الجاهزة: {len(multimodal_batches):,} "
      f"(كل صورة/مقطع يُدرَّب بالاتجاهين معاً: توليد + فهم).")

assert multimodal_batches, "لا توجد دفعات كافية — كبّر MAX_IMAGES/MAX_AUDIO_SAMPLES في خلية جمع البيانات."


### 7) قياس السرعة الحقيقية ثم التدريب الحقيقي

نفس مبدأ المرحلة الأولى بالضبط: نقيس فعلياً بدل التخمين، ثم ندرّب لعدد خطوات واقعي يتناسب مع جلسة 9 ساعات.


In [ ]:
# SHAM_SECTION: sham_small_training_stage2_multimodal
import time
from train import TrainConfig, build_optimizer, train

model.to(device)
# المقياس الثابت قبل أي تحديث للأوزان (حتى قبل قياس السرعة).
from sham_media_link import measure_link, link_report
_media_toks = {"image": image_tokenizer, "audio": audio_tokenizer}
link_before = measure_link(model, tokenizer, _media_toks, FIXED_PAIRS, device)
link_report(link_before)
CALIBRATION_BATCHES = min(10, len(multimodal_batches))
_calib_optimizer = build_optimizer(model, lr=1e-4, weight_decay=0.1)

model.train()
t0 = time.time()
for batch in multimodal_batches[:CALIBRATION_BATCHES]:
    input_ids, labels = batch[0].to(device), batch[1].to(device)
    _, loss = model(input_ids, labels=labels)
    loss.backward()
    _calib_optimizer.step()
    _calib_optimizer.zero_grad()
elapsed = time.time() - t0
steps_per_second = CALIBRATION_BATCHES / elapsed

# على GPU: جلسة أقصر حفاظاً على الرصيد الأسبوعي. على CPU (عند اختياره أو عند نفاد رصيد GPU):
# لا رصيد يُستهلك، فنستغل الجلسة كاملة كباقي المسارات.
MAX_TRAINING_HOURS = 3.5 if device.startswith("cuda") else 8.5
realistic_steps_for_session = int(steps_per_second * MAX_TRAINING_HOURS * 3600 * 0.85)
print(f"سرعة حقيقية مقاسة الآن: {steps_per_second:.3f} خطوة/ثانية على {device}")
print(f"عدد خطوات واقعي ضمن {MAX_TRAINING_HOURS} ساعة: {realistic_steps_for_session:,}")

# دفعات الجلسة: حصة أكبر لأزواج النص↔الصورة/الصوت (كل زوج مرّتان) + نص عربي للتذكّر بحصة أصغر
# حتى لا ينسى اللغة — التفاصيل في sham_media_link.session_batches (تتحسّن من المستودع بلا استيراد).
from sham_media_link import session_batches, measure_link, link_report
from data_acquisition import text_stream_position

GRAD_ACCUM = 4
TEXT_SEQ_LEN = min(512, model.cfg.max_seq_len)
training_batches, _mix_note = session_batches(
    multimodal_batches, tokenizer, TEXT_SEQ_LEN, IMAGE_BATCH_SIZE, realistic_steps_for_session,
    seen_articles=text_stream_position("wikimedia/wikipedia", "20231101.ar"),
)
TOTAL_STEPS = max(len(training_batches) // GRAD_ACCUM, 1)

train_cfg = TrainConfig(
    seq_len=model.cfg.max_seq_len, batch_size=IMAGE_BATCH_SIZE, grad_accum_steps=GRAD_ACCUM, lr=1e-4,
    warmup_steps=max(20, TOTAL_STEPS // 100), total_steps=start_step + TOTAL_STEPS,
    checkpoint_dir="/kaggle/working/checkpoints", checkpoint_every=100, log_every=10,
    max_wall_clock_seconds=MAX_TRAINING_HOURS * 3600,
)
loss_history = train(model, training_batches, train_cfg, device=device, start_step=start_step, resume_optimizer=_calib_optimizer)
print(f"\nانتهى تدريب المرحلة الثانية على {len(loss_history):,} خطوة حقيقية.")
print(f"متوسط الخسارة في أول 10 خطوات: {sum(loss_history[:10]) / min(10, len(loss_history)):.4f}")
print(f"متوسط الخسارة في آخر 10 خطوات: {sum(loss_history[-10:]) / min(10, len(loss_history)):.4f}")

link_after = measure_link(model, tokenizer, _media_toks, FIXED_PAIRS, device)
link_summary = link_report(link_before, link_after)

### 8) حفظ النتيجة النهائية

النموذج الآن يحمل قدرة النص من المرحلة الأولى + قدرة الصورة والصوت (توليداً وفهماً) من هذه المرحلة، في نفس الأوزان. اضغط **Save Version** بعد انتهاء هذه الخلية ليُحفظ كل شيء كـ Output دائم.


In [ ]:
# SHAM_SECTION: sham_small_training_stage2_multimodal
from checkpoint import save_checkpoint

final_step = start_step + len(loss_history)
save_checkpoint("/kaggle/working/checkpoints/final_multimodal.pt", model, final_step)
print(f"تم حفظ النموذج النهائي (نص+صورة+صوت) عند الخطوة {final_step:,}.")
print("أداتا ترميز الصورة والصوت المدرَّبتان محفوظتان أيضاً في /kaggle/working/ (image_tokenizer.pt, audio_tokenizer.pt).")
print("\nاضغط Save Version الآن لحفظ كل هذا كـ Output دائم قابل لإضافته كـ Input لخدمة serve.py أو لتدريب لاحق.")


### 9) نشر النتيجة تلقائياً إلى `sham-multimodal-checkpoint`

منها يستأنف التشغيل القادم لهذه المرحلة، ويُشغَّل خادم شام ودفتر تجربة البوت — بلا تنزيل يدوي.

In [ ]:
# SHAM_SECTION: sham_small_training_stage2_multimodal
import shutil as _shutil
from sham_inputs import publish_dataset

upload_dir = Path("/kaggle/working/for_dataset_upload")
if upload_dir.exists():
    _shutil.rmtree(upload_dir)
upload_dir.mkdir(parents=True)
_shutil.copy2("/kaggle/working/checkpoints/final_multimodal.pt", upload_dir / "final_multimodal.pt")
for _f in ("sham_small_tokenizer.json", "image_tokenizer.pt", "audio_tokenizer.pt"):
    _shutil.copy2(Path("/kaggle/working") / _f, upload_dir / _f)
for _ledger in Path("/kaggle/working/checkpoints").glob("*_ledger.json"):
    _shutil.copy2(_ledger, upload_dir / _ledger.name)
published = publish_dataset(upload_dir, "sham-multimodal-checkpoint", f"stage2 at step {final_step:,}")